<a href="https://colab.research.google.com/github/DanLePoGo/untitled/blob/master/GooglemapsBdeb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Importations
import csv
import pandas as pd
import re
import plotly.graph_objects as go


In [ ]:
#@title Graphe

class Graphe:

    # Graphe représenté par une matrice d'adjacence. On utilise un dictionnaire pour retrouver
    # l'indice d'un sommet
    def __init__(self, dirige=False, value=False):
        self._sommets = {}
        self._matrice = []
        self._dirige = dirige
        self._value = value
        if self._value:
              self._valeur_defaut = None
        else:
            self._valeur_defaut = False

    # Retourne la liste des sommets du graphe dans l'ordre d'insertion
    def get_sommets(self) -> list:
        return list(self._sommets.keys())

    # Retourne vrai si le sommet_2 est adjacent au sommet_1
    def est_adjacent(self, sommet_1, sommet_2) -> bool:
      indice_1 = self._sommets[sommet_1]
      indice_2 = self._sommets[sommet_2]
      return self._matrice[indice_1][indice_2] not in [None, False]

    # Retourne le poids d'un graphe valué
    def get_poids(self, sommet_1, sommet_2):
      indice_1 = self._sommets[sommet_1]
      indice_2 = self._sommets[sommet_2]
      if not self._value:
          raise ValueError("Pas de poids dans un graphe non valué")
      return self._matrice[indice_1][indice_2]

    # Ajoute un sommet au graphe
    def ajouter_sommet(self, sommet):
        if sommet in self._sommets.keys():
            raise ValueError("Le sommet existe déjà")
        else:
            nouvel_indice = len(self._sommets)
            nouveau_tableau = [self._valeur_defaut] * (nouvel_indice + 1)
            self._sommets[sommet] = nouvel_indice
            for ligne in self._matrice:
                ligne.append(self._valeur_defaut)
            self._matrice.append(nouveau_tableau)

    # Ajoute une arete entre le sommet_1 et le sommet_2
    def ajouter_arete(self, sommet_1, sommet_2, poids=None):
        if poids == None and self._value:
            raise ValueError("Il faut un poids dans un graphe valué")
        indice_1 = self._sommets[sommet_1]
        indice_2 = self._sommets[sommet_2]

        if self._value:
            valeur = poids
        else:
            valeur = True

        self._matrice[indice_1][indice_2] = valeur
        if not self._dirige:
            self._matrice[indice_2][indice_1] =  valeur

    # Reetourne la liste de voisins des sommets
    def get_voisins(self, sommet):
        voisins = []
        for candidat in self._sommets.keys():
            if self.est_adjacent(sommet, candidat):
                voisins.append(candidat)
        return voisins



In [ ]:
#@title Algorithme Dijkstra
# Fonction utilisée par Dijkstra pour trouver le sommet ouvert
# le moins cher
def trouver_sommet_moins_cher(moins_cher:dict, non_visites:set):
    valeur = 10**18
    candidat = None

    for sommet in moins_cher.keys():
        if sommet in non_visites and moins_cher[sommet] < valeur:
            valeur = moins_cher[sommet]
            candidat = sommet

    return candidat

# Implantation de dijkstra. Retourne le chemin de depart à
# arrive s'il existe. Le premier sommet de la liste est le sommet
# de depart.
def dijkstra(un_graphe:Graphe, depart, arrive):
    couts = {}
    predecesseurs = {}
    ouverts = set()

    for sommet in un_graphe.get_sommets():
        couts[sommet] = 10**18
        predecesseurs[sommet] = None
        ouverts.add(sommet)

    couts[depart] = 0
    ouverts.remove(depart)
    courant = depart

    while courant != None:
        # On calcule les nouveaux chemins avec le sommet courant
        for voisin in un_graphe.get_voisins(courant):
            if voisin in ouverts:
                if couts[voisin] > couts[courant] + un_graphe.get_poids(courant, voisin):
                    couts[voisin] = couts[courant] + un_graphe.get_poids(courant, voisin)
                    predecesseurs[voisin] = courant

        courant = trouver_sommet_moins_cher(couts, ouverts)
        if courant:
            ouverts.remove(courant)


    chemin = []
    if couts[arrive] != 10**18:
        predecesseur = arrive
        chemin.append(predecesseur)
        while predecesseur != depart:
            predecesseur = predecesseurs[predecesseur]
            chemin.append(predecesseur)

    chemin.reverse()
    distance = couts[arrive]
    return chemin, distance


In [ ]:
#@title Classe Ecole
class Ecole:

    def __init__(self):

      #self._construire_graphe()
      self._position = None
      self._ecole = Graphe(False, True)

    def get_position(self):
      """
      Retourne la position du joueur
      """
      return self._position

    def get_graphe(self): #renvoie le graphe
      return self._ecole

    def _lire_csv(self, nom_fichier:str):
      with open(nom_fichier, newline='', encoding='utf-8') as f:
        lecteur = csv.reader(f)

        next(lecteur)  # skip header

        for ligne in lecteur:
          local_1 = ligne[0]
          local_2 = ligne[1]
          distance = float(ligne[2])

          if local_1.upper() not in self._ecole.get_sommets():
              self._ecole.ajouter_sommet(local_1.upper())

          if local_2 not in self._ecole.get_sommets():
              self._ecole.ajouter_sommet(local_2.upper())
          self._ecole.ajouter_arete(local_1.upper(), local_2.upper(), distance)



In [ ]:
#@title Fonctions
def menu():
  print("-----------------------------------------------------")
  print("Choisir parmi ces options:")
  print()
  print("Trouver le chemin               (T)")
  print()
  print("Entrer une vitesse de marche    (V)")
  print()
  print("Entrer un poids                 (P)")
  print()
  print("Quitter le programme            (Q)")
  print()
  option = input("Entrez votre choix: ").upper()
  return option

def calculate_calories(weight, speed, distance):
    #Estime le nombre de calories brulées selon le poids, la vitesse et la distance


    # Métabolisme de base en fonction de la vitesse de marche
    if speed < 0.89:
        met = 2.0
    elif speed < 1.33:
        met = 3.0
    elif speed < 1.56:
        met = 3.5
    elif speed < 1.78:
        met = 4.3
    else:
        met = 5.0

    # Time = distance / speed
    time_hours = (distance/1000) / (speed*3.6)

    # formule pour calculer les calories
    calories = round((met * float(weight) * float(time_hours)))
    return calories






In [ ]:
#@title Visualisation 3D
def get_floor(node):
    match = re.match(r"S-(\d+)", str(node).upper())
    if not match:
        return 0
    return int(match.group(1)) // 100

def get_room_number(node):
    match = re.match(r"S-(\d+)", str(node).upper())
    if not match:
        return 0
    return int(match.group(1))

def get_suffix(node):
    return get_room_number(node) % 100

def visualiser_maquette_3d(nom_fichier_csv, chemin=None):
    matrice = []

    with open(nom_fichier_csv, "r", encoding="utf-8") as fichier:
        for ligne in csv.reader(fichier):
            matrice.append(ligne)

    entete = matrice.pop(0)
    df = pd.DataFrame(matrice, columns=entete)
    df["distance"] = df["distance"].astype(float)
    df["local1"] = df["local1"].str.upper()
    df["local2"] = df["local2"].str.upper()

    nodes = sorted(set(df["local1"]).union(set(df["local2"])))

    floor_height = 70
    corridor_half_width = 8
    scale_x = 8
    scale_y = 6

    floor_colors = {
        0: "#8d99ae",
        1: "#4dabf7",
        2: "#51cf66",
        3: "#ffd43b",
        4: "#ff6b6b"
    }

    rows = []
    for node in nodes:
        num = get_room_number(node)
        floor = get_floor(node)
        suffix = get_suffix(node)

        x = suffix * scale_x
        y = (corridor_half_width * scale_y) if num % 2 == 0 else -(corridor_half_width * scale_y)
        z = floor * floor_height

        rows.append({
            "local": node,
            "floor": floor,
            "x": x,
            "y": y,
            "z": z
        })

    coords_df = pd.DataFrame(rows)

    xmin = coords_df["x"].min() - 30
    xmax = coords_df["x"].max() + 30
    ymin = coords_df["y"].min() - 15
    ymax = coords_df["y"].max() + 15

    fig = go.Figure()

    # Plaques des étages
    for floor in sorted(coords_df["floor"].unique()):
        z = floor * floor_height

        fig.add_trace(go.Mesh3d(
            x=[xmin, xmax, xmax, xmin],
            y=[ymin, ymin, ymax, ymax],
            z=[z, z, z, z],
            i=[0, 0],
            j=[1, 2],
            k=[2, 3],
            color=floor_colors.get(floor, "#cccccc"),
            opacity=0.18,
            hoverinfo="skip",
            showscale=False,
            name=f"Plaque étage {floor}"
        ))

        fig.add_trace(go.Scatter3d(
            x=[xmin, xmax],
            y=[0, 0],
            z=[z, z],
            mode="lines",
            line=dict(color="black", width=4, dash="dash"),
            hoverinfo="skip",
            showlegend=False
        ))

    # Escaliers
    stair_columns = {
        "Escalier gauche": 76,
        "Escalier centre": 36,
        "Escalier droit": 6
    }

    zmax = coords_df["z"].max()

    for name, x0 in stair_columns.items():
        x0 = x0 * scale_x
        fig.add_trace(go.Scatter3d(
            x=[x0, x0],
            y=[0, 0],
            z=[0, zmax],
            mode="lines+text",
            line=dict(width=8),
            text=[None, name],
            textposition="top center",
            hoverinfo="text",
            showlegend=False
        ))

    # Points des locaux
    for floor in sorted(coords_df["floor"].unique()):
        sub = coords_df[coords_df["floor"] == floor]

        fig.add_trace(go.Scatter3d(
            x=sub["x"],
            y=sub["y"],
            z=sub["z"],
            mode="markers+text",
            text=sub["local"],
            textposition="top center",
            marker=dict(
                size=6,
                color=floor_colors.get(floor, "#333333")
            ),
            name=f"Étage {floor}",
            hovertemplate=(
                "Local: %{text}<br>"
                "Étage: " + str(floor) + "<br>"
                "x: %{x}<br>"
                "y: %{y}<extra></extra>"
            )
        ))

    # Arêtes du graphe
    positions = coords_df.set_index("local")[["x", "y", "z"]].to_dict("index")
    aretes_deja_tracees = set()

    for _, row in df.iterrows():
        a = row["local1"]
        b = row["local2"]
        cle = tuple(sorted([a, b]))

        if cle in aretes_deja_tracees:
            continue
        if a not in positions or b not in positions:
            continue

        aretes_deja_tracees.add(cle)

        fig.add_trace(go.Scatter3d(
            x=[positions[a]["x"], positions[b]["x"]],
            y=[positions[a]["y"], positions[b]["y"]],
            z=[positions[a]["z"], positions[b]["z"]],
            mode="lines",
            line=dict(color="rgba(80,80,80,0.45)", width=4),
            hoverinfo="skip",
            showlegend=False
        ))

    # Chemin optimal surligné
    if chemin and len(chemin) >= 2:
        x_path = []
        y_path = []
        z_path = []
        texte_path = []

        for local in chemin:
            if local in positions:
                x_path.append(positions[local]["x"])
                y_path.append(positions[local]["y"])
                z_path.append(positions[local]["z"])
                texte_path.append(local)

        if len(x_path) >= 2:
            fig.add_trace(go.Scatter3d(
                x=x_path,
                y=y_path,
                z=z_path,
                mode="lines+markers+text",
                text=texte_path,
                textposition="bottom center",
                line=dict(color="red", width=10),
                marker=dict(size=7, color="red"),
                name="Chemin optimal",
                hovertemplate="Local: %{text}<extra></extra>"
            ))

    fig.update_layout(
        title="Maquette 3D interactive de l'école",
        scene=dict(
            xaxis_title="Position dans le corridor",
            yaxis_title="Côté du corridor",
            zaxis_title="Étage / hauteur",
            aspectmode="manual",
            aspectratio=dict(x=3.2, y=1.8, z=2.8),
            camera=dict(eye=dict(x=1.8, y=-1.8, z=1.3))
        ),
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig.show()


In [ ]:
#@title Main
def main():

  ecole = Ecole()
  nom_fichier = input("Entrez le nom complet du fichier csv: (Exemple: graphe_ignace_v2.csv)")
  ecole._lire_csv(nom_fichier)
  graphe = ecole.get_graphe()
  option = ""
  speed = 1.25
  poids = None
  while option != "Q":
    option = menu()
    print()

    #option pour quitter le programme
    if option == "Q":
      print("Merci d'avoir utilisé notre programme de chemin BdeB")
      print("Auteurs: Ilyas Mouadeb, Dan Nguyen, Alejandro Pascual Gomez, Gia-Bao Truong et Christian Wang")

    #option pour trouver un chemin à partir du local de départ et d'arrivée

    elif option == "T":

      depart = input("Entrez votre local de départ: ").upper()
      while depart not in graphe.get_sommets():
        print("Le point de départ n'est pas un local valide")
        depart = input("Entrez votre local de départ: ").upper()
      print()
      arrivé = input("Entrez votre local destination: ").upper()
      while arrivé not in graphe.get_sommets():
        print("Le point d'arrivé n'est pas un local valide")
        arrivé = input("Entrez votre local destination: ").upper()
      print()


      chemin, distance = dijkstra(graphe, depart, arrivé)
      temps = distance/speed
      print("-----------------------------------------------------")
      for i in range(len(chemin)-1):
        print("  marchez et passez devant le local", chemin[i])
        if chemin[i+1][2] > chemin[i][2]:
          print("  prendre l'escalier et montez jusqu'à l'étage", int(chemin[i+1][2]))
        elif chemin[i+1][2] < chemin[i][2]:
          print("  prendre l'escalier et descendez jusqu'à l'étage", int(chemin[i+1][2]))
      print("  marchez et passez devant le local", arrivé)
      print("  Vous êtes arrivés au local", arrivé)
      print("-----------------------------------------------------")
      print()

      print("La distance est de", round(distance, 2), "mètres")
      print("Le trajet vous prendra", int(temps//60), "minutes", int(temps%60), "secondes")
      print()
      if poids is not None:
        calories_burned = calculate_calories(poids, speed, distance)
        print(f"\nCalories brûlées: {calories_burned:.2f} calories")

      visualiser_maquette_3d(nom_fichier, chemin) #Affichage


    elif option == "V":
      vitesse = input("Entrez une vitesse de marche: Sportive(S), Rapide(R), Normale(N), Lente(L): ").upper()

      if vitesse == "S":
        speed = 1.95
      elif vitesse == "R":
        speed = 1.53
      elif vitesse == "N":
        speed = 1.25
      elif vitesse == "L":
        speed = 0.97

    elif option == "P":
      poids = input("Entrez un poids (en kg): ")




if __name__ == "__main__":
    main()

KeyboardInterrupt: Interrupted by user